# Capstone flow — quiz answers by A+ domain

**Flow statement:** My flow reads `primary_history.csv`, groups answers by A+ domain, flags domains under 70% correct, and a person reviews before anything happens.

**Safety statement:** The agent must never message a learner or change their study plan without approval.

**Expected results** (also asserted in Step 7): rows 8 → 6 kept, dropped 2. Averages: 1.0 Mobile devices 0.50, 2.0 Networking 1.00, 3.0 Hardware 0.50. Flagged (< 0.70): 1.0 Mobile devices and 3.0 Hardware.

**Mapping provenance:** the objective → domain table below was generated from the repo's `content/objectives.json` on 2026-11-19. It covers **CompTIA A+ Core 1 (220-1201)** and **Core 2 (220-1202)**. The mock rows are scoped to Core 1.


In [ ]:
# Step 1 — build the mock table and save it as a CSV
import csv

# Mock rows scoped to Core 1 (220-1201), in the export's column order:
# questionId, objective, correct, timestamp
MOCK_ROWS = [
    {"questionId": "q-m01", "objective": "1.1", "correct": True,  "timestamp": "2026-11-17T21:04:00.000Z"},
    {"questionId": "q-m02", "objective": "1.3", "correct": False, "timestamp": "2026-11-17T21:05:00.000Z"},
    {"questionId": "q-m03", "objective": "2.1", "correct": True,  "timestamp": "2026-11-17T21:06:00.000Z"},
    {"questionId": "q-m04", "objective": "2.5", "correct": True,  "timestamp": "2026-11-17T21:07:00.000Z"},
    {"questionId": "q-m05", "objective": "3.2", "correct": True,  "timestamp": "2026-11-17T21:08:00.000Z"},
    {"questionId": "q-m06", "objective": "3.5", "correct": False, "timestamp": "2026-11-17T21:09:00.000Z"},
    # two broken rows on purpose: one empty correct, one word in correct
    {"questionId": "q-m07", "objective": "1.2", "correct": "",     "timestamp": "2026-11-17T21:10:00.000Z"},
    {"questionId": "q-m08", "objective": "2.2", "correct": "none", "timestamp": "2026-11-17T21:11:00.000Z"},
]

CSV_PATH = "primary_history.csv"

with open(CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["questionId", "objective", "correct", "timestamp"])
    writer.writeheader()
    writer.writerows(MOCK_ROWS)

print(f"saved {CSV_PATH}")
print(open(CSV_PATH).read())


In [ ]:
# Step 2 — inspect the file
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(df.head())
print(df.shape)
df.info()
print(df.describe(include="all"))


In [ ]:
# Step 3 — required columns, checked with a set difference against REQUIRED
REQUIRED = {"questionId", "objective", "correct", "timestamp"}

def require_columns(rows):
    if rows:
        missing = REQUIRED - set(rows[0])
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")

require_columns(MOCK_ROWS)

# a bad file that is missing the correct column, shown inside try/except
# so this cell stays green
BAD_PATH = "primary_history_bad.csv"
with open(BAD_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["questionId", "objective", "timestamp"])
    writer.writeheader()
    writer.writerow({"questionId": "q-x1", "objective": "3.1", "timestamp": "2026-11-17T21:12:00.000Z"})

with open(BAD_PATH, newline="") as f:
    bad_rows = list(csv.DictReader(f))

try:
    require_columns(bad_rows)
except ValueError as error:
    print(error)


In [ ]:
# Step 4 — convert correct to 0/1 and drop rows we cannot trust
# Read back from the CSV, True/False arrive as text ("True"/"False"), and
# pd.to_numeric alone would turn every row into NaN — so we map the two real
# values explicitly and count everything else as missing.

TRUE_TEXT = {"true", "1"}
FALSE_TEXT = {"false", "0"}

def clean_answers(rows):
    kept, missing = [], 0
    for row in rows:
        text = str(row["correct"]).strip().lower()
        if text in TRUE_TEXT:
            correct = 1        # True/"true"/"1" -> 1
        elif text in FALSE_TEXT:
            correct = 0        # False/"false"/"0" -> 0
        else:
            missing += 1       # empty string or a word like "none": not a real answer
            continue
        kept.append({**row, "correct": correct})
    return kept, missing

with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))

kept, missing = clean_answers(rows)
print(f"missing correct after conversion: {missing}")
print(f"rows: {len(rows)} -> {len(kept)} | dropped: {missing}")


## Step 4 — convert + clean (handling rules)

The CSV round-trip stores `True`/`False` as text, so the two real values are mapped to 1/0. **Handling rules, stated before the code:** an empty `correct` counts as missing, and any other value (like the word `none`) is not a real answer and also counts as missing — every row that fails the conversion is dropped and counted in `dropped_count`. Nothing is guessed or filled in: dropped rows are tolerated and reported, never silently fixed (cleaning rule from the May academy data-pipeline class). The same principle drives the wrapper's missing-file fallback in Step 6: the flow tolerates a missing file by serving a one-row mock and `dropped_count: 0`, but the condition itself stays visible.

## Step 5 — summarize by domain and flag

Note on flagging: the guide's starter flags rows with a `>=` mask; this flow flags **whole domains whose average falls below 70%**, because the human-review decision is about where to focus study time. (Asking the instructor whether this satisfies "Make it yours.")


In [ ]:
# Step 5 — map objectives to A+ domains, average correct per domain, flag below 70%
# Objective -> domain mapping, generated from the repo's content/objectives.json
# on 2026-11-19. Covers CompTIA A+ Core 1 (220-1201) and Core 2 (220-1202).
OBJECTIVE_DOMAIN = {
    "220-1201 1.1": "1.0 Mobile devices",
    "220-1201 1.2": "1.0 Mobile devices",
    "220-1201 1.3": "1.0 Mobile devices",
    "220-1201 2.1": "2.0 Networking",
    "220-1201 2.2": "2.0 Networking",
    "220-1201 2.3": "2.0 Networking",
    "220-1201 2.4": "2.0 Networking",
    "220-1201 2.5": "2.0 Networking",
    "220-1201 2.6": "2.0 Networking",
    "220-1201 2.7": "2.0 Networking",
    "220-1201 2.8": "2.0 Networking",
    "220-1201 3.1": "3.0 Hardware",
    "220-1201 3.2": "3.0 Hardware",
    "220-1201 3.3": "3.0 Hardware",
    "220-1201 3.4": "3.0 Hardware",
    "220-1201 3.5": "3.0 Hardware",
    "220-1201 3.6": "3.0 Hardware",
    "220-1201 3.7": "3.0 Hardware",
    "220-1201 3.8": "3.0 Hardware",
    "220-1201 4.1": "4.0 Virtualization and cloud computing",
    "220-1201 4.2": "4.0 Virtualization and cloud computing",
    "220-1201 5.1": "5.0 Hardware and network troubleshooting",
    "220-1201 5.2": "5.0 Hardware and network troubleshooting",
    "220-1201 5.3": "5.0 Hardware and network troubleshooting",
    "220-1201 5.4": "5.0 Hardware and network troubleshooting",
    "220-1201 5.5": "5.0 Hardware and network troubleshooting",
    "220-1201 5.6": "5.0 Hardware and network troubleshooting",
    "220-1202 1.1": "1.0 Operating systems",
    "220-1202 1.2": "1.0 Operating systems",
    "220-1202 1.3": "1.0 Operating systems",
    "220-1202 1.4": "1.0 Operating systems",
    "220-1202 1.5": "1.0 Operating systems",
    "220-1202 1.6": "1.0 Operating systems",
    "220-1202 1.7": "1.0 Operating systems",
    "220-1202 1.8": "1.0 Operating systems",
    "220-1202 1.9": "1.0 Operating systems",
    "220-1202 1.10": "1.0 Operating systems",
    "220-1202 1.11": "1.0 Operating systems",
    "220-1202 2.1": "2.0 Security",
    "220-1202 2.2": "2.0 Security",
    "220-1202 2.3": "2.0 Security",
    "220-1202 2.4": "2.0 Security",
    "220-1202 2.5": "2.0 Security",
    "220-1202 2.6": "2.0 Security",
    "220-1202 2.7": "2.0 Security",
    "220-1202 2.8": "2.0 Security",
    "220-1202 2.9": "2.0 Security",
    "220-1202 2.10": "2.0 Security",
    "220-1202 2.11": "2.0 Security",
    "220-1202 3.1": "3.0 Software troubleshooting",
    "220-1202 3.2": "3.0 Software troubleshooting",
    "220-1202 3.3": "3.0 Software troubleshooting",
    "220-1202 3.4": "3.0 Software troubleshooting",
    "220-1202 4.1": "4.0 Operational procedures",
    "220-1202 4.2": "4.0 Operational procedures",
    "220-1202 4.3": "4.0 Operational procedures",
    "220-1202 4.4": "4.0 Operational procedures",
    "220-1202 4.5": "4.0 Operational procedures",
    "220-1202 4.6": "4.0 Operational procedures",
    "220-1202 4.7": "4.0 Operational procedures",
    "220-1202 4.8": "4.0 Operational procedures",
    "220-1202 4.9": "4.0 Operational procedures",
    "220-1202 4.10": "4.0 Operational procedures",
}

EXAM = "220-1201"  # the mock rows are Core 1

def map_domain(row):
    if row["objective"] in (None, ""):
        return None
    return OBJECTIVE_DOMAIN.get(f"{EXAM} {row['objective']}")

def summarize(kept):
    totals = {}
    for row in kept:
        domain = map_domain(row)
        if domain is None:
            continue  # an objective the mapping cannot place is not summarized
        stats = totals.setdefault(domain, {"sum": 0, "count": 0})
        stats["sum"] += row["correct"]
        stats["count"] += 1
    return [
        {"domain": domain, "average": stats["sum"] / stats["count"], "rows": stats["count"]}
        for domain, stats in sorted(totals.items())
    ]

THRESHOLD = 0.70

def flag_below(summary, threshold=THRESHOLD):
    return [record for record in summary if record["average"] < threshold]

summary = summarize(kept)
flagged = flag_below(summary)
for record in summary:
    print(f"{record['domain']}: {record['average']:.2f} over {record['rows']} rows")
print("flagged (< 0.70):", ", ".join(r["domain"] for r in flagged) or "none")


In [ ]:
# Step 6 — one wrapper the reviewer calls
import os

FALLBACK_ROW = [{
    "questionId": "q-fallback", "objective": "2.1", "correct": True,
    "timestamp": "2026-11-17T21:12:00.000Z",
}]

def analyze_answers(path, threshold=THRESHOLD):
    """Run the whole flow. A missing file falls back to one mock row (dropped_count 0);
    a missing column still raises."""
    if os.path.exists(path):
        with open(path, newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        rows = FALLBACK_ROW
    require_columns(rows)
    kept, missing = clean_answers(rows)
    summary = summarize(kept)
    return {
        "summary": summary,
        "flagged": flag_below(summary, threshold),
        "dropped_count": missing,
    }


In [ ]:
# Step 7 — four tests, then the all-clear line
result = analyze_answers(CSV_PATH)
assert result == {
    "summary": [
        {"domain": "1.0 Mobile devices", "average": 0.5, "rows": 2},
        {"domain": "2.0 Networking", "average": 1.0, "rows": 2},
        {"domain": "3.0 Hardware", "average": 0.5, "rows": 2},
    ],
    "flagged": [
        {"domain": "1.0 Mobile devices", "average": 0.5, "rows": 2},
        {"domain": "3.0 Hardware", "average": 0.5, "rows": 2},
    ],
    "dropped_count": 2,
}
print("test 1 ok: mock CSV")

fallback = analyze_answers("no-such-file.csv")
assert fallback["dropped_count"] == 0
assert len(fallback["summary"]) == 1
assert fallback["flagged"] == []
print("test 2 ok: missing file ->", fallback)

try:
    analyze_answers(BAD_PATH)
except ValueError as error:
    assert "correct" in str(error)
    print("test 3 ok: missing column ->", error)

assert analyze_answers(CSV_PATH, threshold=0)["flagged"] == []
print("test 4 ok: huge threshold flags nothing")

print("all tests passed")


## Step 8 — the review pause (guide's Step 8 pattern)

`MemorySaver` + `interrupt_before=["human_review"]` + `update_state`. Only the node functions are ours — they run the quiz flow. The state holds **rows as a list of dicts, not a DataFrame**.


In [ ]:
%pip install -q -U langgraph
import langgraph
print("langgraph ready")


In [ ]:
from typing import TypedDict
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

class FlowState(TypedDict):
    rows: list
    summary: list
    flagged: list
    dropped_count: int
    decision: str

def load_node(state):
    with open(CSV_PATH, newline="") as f:
        rows = list(csv.DictReader(f))
    return {"rows": rows}

def clean_node(state):
    kept, missing = clean_answers(state["rows"])
    return {"rows": kept, "dropped_count": missing}

def summarize_node(state):
    summary = summarize(state["rows"])
    flagged = flag_below(summary)
    print("flagged so far:", ", ".join(r["domain"] for r in flagged) or "none")
    return {"summary": summary, "flagged": flagged}

def human_review_node(state):
    print("summary:", ", ".join(f"{r['domain']} {r['average']:.2f}" for r in state["summary"]))
    print("flagged:", ", ".join(r["domain"] for r in state["flagged"]) or "none")
    print("dropped:", state["dropped_count"])
    return {"decision": state["decision"]}

graph = StateGraph(FlowState)
graph.add_node("load", load_node)
graph.add_node("clean", clean_node)
graph.add_node("summarize", summarize_node)
graph.add_node("human_review", human_review_node)
graph.set_entry_point("load")
graph.add_edge("load", "clean")
graph.add_edge("clean", "summarize")
graph.add_edge("summarize", "human_review")
graph.add_edge("human_review", END)

app = graph.compile(checkpointer=MemorySaver(), interrupt_before=["human_review"])


In [ ]:
config = {"configurable": {"thread_id": "run-1"}}

print("--- run until the pause")
app.invoke({}, config)
print("waiting at:", app.get_state(config).next)

print("--- a person approves; resume")
app.update_state(config, {"decision": "approved by reviewer"})
result = app.invoke(None, config)
print("final decision:", result["decision"])


## How to run

Colab: File → Upload notebook → `flow.ipynb`, then File → Save a copy in Drive → Share → "Anyone with the link", Viewer. Run all cells top to bottom; the first run installs `langgraph`.

Full expected outputs and limitations are in `capstone/README.md`.